## 환경설정 및 Jupyter 설치
### 가상환경 세팅
```bash
conda create --name env-langgraph-fc python=3.10 -y
```
<br/>

### 세팅 확인
가상환경 디렉터리 확인
```bash
ls -al ~/miniforge3/envs/env-langgraph-fc 
```
<br/>

가상환경 활성화
```bash
conda activate env-langgraph-fc
```
<br/>

가상환경을 끌때는 다음과 같이 한다.
```bash
conda deactivate
```
<br/>

### 의존성 설정
먼저 jupyter 설치
```bash
# pip 대신 mamba를 사용하면 패키지 설치 속도가 훨씬 빠릅니다.
mamba install -c conda-forge jupyter notebook ipykernel -y
```
<br/>

다음 의존성 설치
```bash
## pip 를 이용해 설치
!pip install langgraph langchain langchain_google_genai langchain_community

## 또는 mamba 를 이용해 설치
mamba install langgraph langchain langchain_google_genai langchain_community
```
<br/>

keyring 라이브러리 설치
```bash
## pip 을 이용해 설치
pip install keyring

## 또는 mamba 를 이용해 keyring 설치
mamba install keyring
```
<br/>

## keyring import (api key 설정)
- (1) 미리 터미널에 입력해둔다.
- (2) python 코드 내에서 사용한다.

```bash
## bash 쉘 에서 다음 내용을 입력
## 형식 keyring set {{서비스명}} {{계정명}}

## e.g.
keyring set gemini-api-key---alpha300uk alpha300uk  
Password for 'alpha300uk' in 'gemini-api-key---alpha300uk':
```

In [1]:
import keyring
service_name = "gemini-api-key---alpha300uk"
username = "alpha300uk"
api_token = keyring.get_password(service_name, username)

## 의존성 설치 (혹시 설치 안했을 경우를 위해 추가한 섹션)

In [ ]:
! pip install langgraph langchain langchain_google_genai langchain_community

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typi

## llm, ratelimiter 선언

In [3]:
import os
os.environ['GOOGLE_API_KEY'] = api_token

from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_google_genai import ChatGoogleGenerativeAI

# Gemini API는 분당 10개 요청으로 제한
# 즉, 초당 약 0.167개 요청 (10/60)
rate_limiter = InMemoryRateLimiter(
    # requests_per_second=0.167,  # 분당 10개 요청
    requests_per_second=1,  # 초당 최대 1개, 분당 최대 60개 요청
    check_every_n_seconds=0.1,  # 100ms마다 체크
    max_bucket_size=10,  # 최대 버스트 크기
)

# rate limiter를 LLM에 적용
llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    rate_limiter=rate_limiter,
    # temperature
    # max_tokens

    thinking_budget = 500  # 추론(Reasoning) 토큰 길이 제한
)

1. `os.environ`: 환경 변수에 Google API 키를 설정합니다.
2. `InMemoryRateLimiter`: Gemini 무료 티어의 제한(분당 10회)을 준수하기 위해 초당 약 0.167회로 요청 속도를 제한하는 객체를 생성합니다.
3. `ChatGoogleGenerativeAI`: `gemini-3-flash-preview` 모델을 초기화하며, 앞서 만든 `rate_limiter`를 적용하여 API 호출 안정성을 확보합니다. `thinking_budget`은 모델의 추론 프로세스에 할당할 토큰 한도를 설정합니다.

## e.g. 구조화된 출력 (1)
- 자료구조 `Objective` 생성 및 정의
- `llm.with_structured_output(Objective)`을 활용해 `Objective` 에 대해 구조화된 출력 생성

In [4]:
from pydantic import BaseModel, Field

# 프롬프트 자동 생성을 위한 요소 저장
class Objective(BaseModel):
    instruction: str = Field(description='프롬프트의 지시 사항을 명확히 재구성')
    output_format: str = Field(description='출력 포맷에 대한 설명')
    examples: str = Field(description='예시 출력(1개)')
    notes: str = Field(description='작업 과정에서 중요한 내용을 4개의 개조식 문장으로 구성')

    @property #
    def as_str(self) -> str:
        return '\n\n'.join([f'## {key}\n {value}' for key, value in self])


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate([
    ('system', '아래의 작업을 보다 자세하게 요청하는 시스템 프롬프트를 구성하고자 합니다. 주어진 포맷에 적절하게 작성하세요.'),
    ('user', '{instruction}')

])

chain = prompt | llm.with_structured_output(Objective)

result = chain.invoke("집에서 쉽게 구할 수 있는 재료로 재미있는 장난감 만들기")

result

Objective(instruction='집에서 흔히 구할 수 있는 재료들을 활용하여 아이들이 만들고 가지고 놀 수 있는 재미있고 창의적인 장난감 제작 방법을 상세하게 설명하세요. 지침은 명확하고 따라하기 쉬워야 하며, 안전을 최우선으로 고려해야 합니다.', output_format='다음과 같은 JSON 형식으로 출력합니다: {"장난감_이름": "[장난감 이름]", "재료": ["재료1", "재료2", "재료3"], "만드는_방법": ["1. 첫 번째 단계", "2. 두 번째 단계", "3. 세 번째 단계"], "안전_수칙": ["안전 수칙1", "안전 수칙2"], "팁": ["추가 팁1", "추가 팁2"]}', examples='{"장난감_이름": "휴지심 망원경", "재료": ["휴지심 1개", "색종이 또는 포장지", "가위", "풀 또는 양면테이프", "끈 (선택 사항)"], "만드는_방법": ["1. 휴지심의 한쪽 끝을 약 1cm 정도 길이로 4등분하여 가위로 자릅니다.", "2. 잘린 부분을 바깥쪽으로 살짝 접어 별 모양을 만듭니다.", "3. 색종이 또는 포장지로 휴지심 전체를 감싸고 풀 또는 테이프로 고정합니다.", "4. 망원경을 꾸미고 싶다면 스티커를 붙이거나 그림을 그립니다.", "5. 끈을 달아 목에 걸 수 있는 망원경을 만들 수도 있습니다."], "안전_수칙": ["가위를 사용할 때는 항상 보호자의 지도를 받으세요.", "풀이나 테이프를 입에 넣지 않도록 주의하세요."], "팁": ["색종이 대신 물감이나 크레용으로 휴지심을 칠해도 좋습니다.", "여러 개의 휴지심을 연결하여 더 긴 망원경을 만들 수 있습니다."]}', notes='- 제시된 장난감은 반드시 집에서 쉽게 찾을 수 있는 재료(예: 휴지심, 종이, 끈, 병뚜껑 등)로 만들 수 있어야 합니다.- 만드는 방법은 단계별로 명확하고 간결하게 설명하여 아이들도 따라하기 쉽게 구성해야 합니다.- 각 장난감에 대한 안전 수칙을 반드시 포함하여 잠재적인 위험을 사전에 방지해야 합니다.-

출력 예시 (출력이 나타나는 데까지 11초 가량 걸림)
```plain
Objective(instruction='사용자가 제시한 주제에 따라, 집에서 쉽게 구할 수 있는 재료만을 활용하여 만들 수 있는 재미있고 창의적인 장난감 만들기 아이디어를 제공하세요. 장난감의 이름, 필요한 재료 목록, 단계별 제작 방법, 그리고 안전 수칙을 명확하고 상세하게 설명해야 합니다.', output_format="출력은 JSON 객체 형식이어야 합니다. 객체는 'toy_name'(장난감 이름), 'materials'(필요한 재료 목록), 'instructions'(단계별 제작 방법), 'safety_tips'(안전 수칙)의 4가지 키를 포함해야 합니다. 'materials'와 'instructions'는 각 항목이 리스트 형태의 문자열로 구성되어야 합니다. 'safety_tips'는 3개 이상의 개조식 문장으로 구성된 문자열입니다.", examples='{"toy_name": "휴지심 망원경", "materials": ["휴지심 2개", "색종이 또는 포장지", "가위", "풀 또는 테이프", "끈 (선택 사항)"], "instructions": ["1. 휴지심 한쪽 끝을 약 1cm 정도 잘라 4등분으로 가위집을 냅니다.", "2. 다른 휴지심의 한쪽 끝을 1번과 동일하게 가위집을 냅니다.", "3. 두 휴지심의 가위집 낸 부분을 서로 교차시켜 끼워 고정합니다. (이때, 풀이나 테이프를 사용하여 더욱 단단히 고정할 수 있습니다.)", "4. 완성된 망원경 몸통에 색종이나 포장지를 붙여 예쁘게 꾸며줍니다.", "5. (선택 사항) 양쪽에 구멍을 뚫어 끈을 연결하면 목에 걸고 다닐 수 있는 망원경이 됩니다."], "safety_tips": "- 가위 사용 시 보호자의 감독 하에 주의하여 사용하세요.- 풀 또는 테이프 사용 후에는 손을 깨끗이 씻으세요.- 작은 부품이 떨어지지 않도록 튼튼하게 만드세요.- 만든 장난감을 입에 넣지 않도록 주의하세요."}', notes='- 제시된 주제에 충실하게 장난감 아이디어를 구체화해야 합니다.- 모든 재료는 가정에서 쉽게 찾을 수 있는 것들로 한정해야 합니다.- 제작 방법은 초등학생도 이해할 수 있도록 간단하고 명확한 단계로 구성해야 합니다.- 장난감의 안전한 사용을 위한 실질적인 안전 수칙을 반드시 포함해야 합니다.')
```

깔끔하게 출력

In [ ]:
print(result.as_str)

## instruction
 집에서 흔히 구할 수 있는 재료들을 활용하여 아이들이 만들고 가지고 놀 수 있는 재미있고 창의적인 장난감 제작 방법을 상세하게 설명하세요. 지침은 명확하고 따라하기 쉬워야 하며, 안전을 최우선으로 고려해야 합니다.

## output_format
 다음과 같은 JSON 형식으로 출력합니다: {"장난감_이름": "[장난감 이름]", "재료": ["재료1", "재료2", "재료3"], "만드는_방법": ["1. 첫 번째 단계", "2. 두 번째 단계", "3. 세 번째 단계"], "안전_수칙": ["안전 수칙1", "안전 수칙2"], "팁": ["추가 팁1", "추가 팁2"]}

## examples
 {"장난감_이름": "휴지심 망원경", "재료": ["휴지심 1개", "색종이 또는 포장지", "가위", "풀 또는 양면테이프", "끈 (선택 사항)"], "만드는_방법": ["1. 휴지심의 한쪽 끝을 약 1cm 정도 길이로 4등분하여 가위로 자릅니다.", "2. 잘린 부분을 바깥쪽으로 살짝 접어 별 모양을 만듭니다.", "3. 색종이 또는 포장지로 휴지심 전체를 감싸고 풀 또는 테이프로 고정합니다.", "4. 망원경을 꾸미고 싶다면 스티커를 붙이거나 그림을 그립니다.", "5. 끈을 달아 목에 걸 수 있는 망원경을 만들 수도 있습니다."], "안전_수칙": ["가위를 사용할 때는 항상 보호자의 지도를 받으세요.", "풀이나 테이프를 입에 넣지 않도록 주의하세요."], "팁": ["색종이 대신 물감이나 크레용으로 휴지심을 칠해도 좋습니다.", "여러 개의 휴지심을 연결하여 더 긴 망원경을 만들 수 있습니다."]}

## notes
 - 제시된 장난감은 반드시 집에서 쉽게 찾을 수 있는 재료(예: 휴지심, 종이, 끈, 병뚜껑 등)로 만들 수 있어야 합니다.- 만드는 방법은 단계별로 명확하고 간결하게 설명하여 아이들도 따라하기 쉽게 구성해야 합니다.- 각 장난감에 대한 안전 수칙을 반드시 포함하여 잠재적인 위험을 사전에 방지해야 합니다.- 

출력 예시
```plain
## instruction
 사용자가 제시한 주제에 따라, 집에서 쉽게 구할 수 있는 재료만을 활용하여 만들 수 있는 재미있고 창의적인 장난감 만들기 아이디어를 제공하세요. 장난감의 이름, 필요한 재료 목록, 단계별 제작 방법, 그리고 안전 수칙을 명확하고 상세하게 설명해야 합니다.

## output_format
 출력은 JSON 객체 형식이어야 합니다. 객체는 'toy_name'(장난감 이름), 'materials'(필요한 재료 목록), 'instructions'(단계별 제작 방법), 'safety_tips'(안전 수칙)의 4가지 키를 포함해야 합니다. 'materials'와 'instructions'는 각 항목이 리스트 형태의 문자열로 구성되어야 합니다. 'safety_tips'는 3개 이상의 개조식 문장으로 구성된 문자열입니다.

## examples
 {"toy_name": "휴지심 망원경", "materials": ["휴지심 2개", "색종이 또는 포장지", "가위", "풀 또는 테이프", "끈 (선택 사항)"], "instructions": ["1. 휴지심 한쪽 끝을 약 1cm 정도 잘라 4등분으로 가위집을 냅니다.", "2. 다른 휴지심의 한쪽 끝을 1번과 동일하게 가위집을 냅니다.", "3. 두 휴지심의 가위집 낸 부분을 서로 교차시켜 끼워 고정합니다. (이때, 풀이나 테이프를 사용하여 더욱 단단히 고정할 수 있습니다.)", "4. 완성된 망원경 몸통에 색종이나 포장지를 붙여 예쁘게 꾸며줍니다.", "5. (선택 사항) 양쪽에 구멍을 뚫어 끈을 연결하면 목에 걸고 다닐 수 있는 망원경이 됩니다."], "safety_tips": "- 가위 사용 시 보호자의 감독 하에 주의하여 사용하세요.- 풀 또는 테이프 사용 후에는 손을 깨끗이 씻으세요.- 작은 부품이 떨어지지 않도록 튼튼하게 만드세요.- 만든 장난감을 입에 넣지 않도록 주의하세요."}

## notes
 - 제시된 주제에 충실하게 장난감 아이디어를 구체화해야 합니다.- 모든 재료는 가정에서 쉽게 찾을 수 있는 것들로 한정해야 합니다.- 제작 방법은 초등학생도 이해할 수 있도록 간단하고 명확한 단계로 구성해야 합니다.- 장난감의 안전한 사용을 위한 실질적인 안전 수칙을 반드시 포함해야 합니다.
```

## e.g. (2)

In [ ]:
### 예전 코드 --- (start)
import keyring
import os

from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_google_genai import ChatGoogleGenerativeAI

service_name = "gemini-api-key---alpha300uk"
username = "alpha300uk"

# 키링에서 토큰 조회
api_token = keyring.get_password(service_name, username)

# api_token 등록
os.environ['GOOGLE_API_KEY'] = api_token

# Gemini API는 분당 10개 요청으로 제한
# 즉, 초당 약 0.167개 요청 (10/60)
rate_limiter = InMemoryRateLimiter(
    # requests_per_second=0.167,  # 분당 10개 요청
    requests_per_second=1,  # 초당 최대 1개, 분당 최대 60개 요청
    check_every_n_seconds=0.1,  # 100ms마다 체크
    max_bucket_size=10,  # 최대 버스트 크기
)

# rate limiter를 LLM에 적용
llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    rate_limiter=rate_limiter,
    # temperature
    # max_tokens

    thinking_budget = 500  # 추론(Reasoning) 토큰 길이 제한
)

# 프롬프트 자동 생성을 위한 요소 저장
from pydantic import BaseModel, Field
class Objective(BaseModel):
    instruction: str = Field(description='프롬프트의 지시 사항을 명확히 재구성')
    output_format: str = Field(description='출력 포맷에 대한 설명')
    examples: str = Field(description='예시 출력(1개)')
    notes: str = Field(description='작업 과정에서 중요한 내용을 4개의 개조식 문장으로 구성')

    @property #
    def as_str(self) -> str:
        return '\n\n'.join([f'## {key}\n {value}' for key, value in self])


# from langchain_core.prompts import ChatPromptTemplate
# prompt = ChatPromptTemplate([
#     ('system', '아래의 작업을 보다 자세하게 요청하는 시스템 프롬프트를 구성하고자 합니다. 주어진 포맷에 적절하게 작성하세요.'),
#     ('user', '{instruction}')

# ])
### 예전 코드 --- (end)

from typing import TypedDict
class State(TypedDict):
    instruction : str
    prompt_materials : Objective # Objective Class를 하나의 값에 저장
    full_prompt : str
    result : str


from langchain_core.prompts import ChatPromptTemplate

"""
(1) 'instruction' 속성이 포함된 State 객체를 state 를 받아서 (참고: graph.stream)
(2) chain : llm 을 거친 후 llm 이 Object 타입으로 structured_output(구조화된 출력)으로 변환하는 파이프라이닝 정의
(3) result : llm 을 통해 프롬프트를 생성하며 Objective 타입으로 구조화된 프롬프트 객체 'result'를 생성
    llm.with_structured_output(Objective) 은 llm 이 Objective 타입으로 결과물을 변환해서 반환합니다.
(4) ret_result : 결과값으로는 조금 더 풍부해진 프롬프트인 {'prompt_materials': result} 를 return 하며, prompt_materials 는 Objective 타입입니다.
"""
def get_prompt_materials(State):
    prompt = ChatPromptTemplate([
        ('system', '아래의 작업을 보다 자세하게 세분화하고자 합니다. 주어진 포맷에 적절하게 작성하세요.'),
        ('user', '{instruction}')

    ])

    chain = prompt | llm.with_structured_output(Objective)

    result = chain.invoke({'instruction':State['instruction']})
    print(f"(get_prompt_materials) 'instruction' = {State['instruction']}, 생성된 프롬프트 생성 질문(생성재료 'prompt_materials') {result}")
    
    ret_result = {'prompt_materials' : result}
    # print(f"💡 ret_result = {ret_result}")
    return ret_result


"""
(1) {'prompt_materials': Objective} 속성이 포함된 State 객체를 state 를 받아서 ChatPromptTemplate 객체 prompt를 만든 후
(2) chain : llm을 거친 후 llm 이 StrOutputParser()를 통해 str 타입으로 변환하는 파이프라이닝 정의
(3) result : 실행 결과는 str 타입 (StrOutputParser 를 거쳤으므로)
(4) ret_result : {'full_prompt': result} 로 감싸서 'full_prompt' 속성에 대해 바인딩
"""
from langchain_core.output_parsers import StrOutputParser
def generate_prompt(State):
    prompt = ChatPromptTemplate([
        ('system', '''당신은 체계적이고 정확한 프롬프트 엔지니어입니다. 아래의 포인트를 바탕으로, LLM에 입력할  시스템 프롬프트를 작성하세요.
{points}'''),
        ('user', '{instruction}')
    ])
    

    chain = prompt | llm | StrOutputParser()

    ## 참고) 
    #### result : result 는 StrOutputParser() 를 통해 str 타입으로 변환된 결과물
    #### StrOutputParser : AI가 준 복잡한 객체에서 다른 군더더기(토큰 정보 등)는 다 버리고, 실제 답변인 '문자열(String)'만 추출합니다.
    result = chain.invoke({'instruction': State['instruction'], 'points': State['prompt_materials'].as_str})
    print(f"(generate_prompt) State = {State}, 생성된 프롬프트 결과물 {result}")
    ret_result = {'full_prompt' : result}
    # print(f"💡 ret_result = {ret_result}")
    return ret_result


def generate(State):
    return {'result' : llm.invoke(State['full_prompt']).content}


from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END


# 그래프 구성
builder = StateGraph(State)

builder.add_node("get_prompt_materials", get_prompt_materials)
builder.add_node("generate_prompt", generate_prompt)
builder.add_node("generate", generate)

builder.add_edge(START, "get_prompt_materials")
builder.add_edge("get_prompt_materials", "generate_prompt")
builder.add_edge("generate_prompt", "generate")

builder.add_edge("generate", END)
graph = builder.compile()


import pprint
# Streaming 참고
# https://langchain-ai.github.io/langgraph/concepts/streaming/#streaming-graph-outputs-stream-and-astream

for data in graph.stream({'instruction': '''영화 '마이너리티 리포트'와 AI 윤리의 연관성에 대한 리포트 쓰기'''},
                         stream_mode='values'):
    pprint.pprint(data)
    print('----')

{'instruction': "영화 '마이너리티 리포트'와 AI 윤리의 연관성에 대한 리포트 쓰기"}
----
(get_prompt_materials) 'instruction' = 영화 '마이너리티 리포트'와 AI 윤리의 연관성에 대한 리포트 쓰기, 생성된 프롬프트 생성 질문(생성재료 'prompt_materials') instruction="영화 '마이너리티 리포트'의 주요 내용을 분석하고, 이 영화가 다루는 선지적 사법 시스템이 현대 AI 기술의 발전과 관련하여 제기하는 윤리적 쟁점들을 심층적으로 탐구하여 리포트를 작성합니다. 특히 예측 기술의 정확성, 개인의 자유 침해, 책임 소재 문제 등을 중심으로 다룹니다." output_format='서론, 본론(영화 분석, AI 윤리 쟁점), 결론의 구조를 갖춘 리포트 형식으로 작성되어야 합니다. 각 섹션은 명확한 소제목과 함께 논리적으로 전개되어야 하며, 전문적이고 설득력 있는 문체로 구성합니다.' examples="서론: 영화 '마이너리티 리포트'는 2054년 워싱턴 D.C.를 배경으로, 범죄가 발생하기 전 예측하여 범인을 체포하는 '프리크라임' 시스템을 통해 AI 기반 예측 기술의 윤리적 딜레마를 심도 있게 다룹니다. 본 리포트는 영화 속 예측 사법 시스템을 분석하고, 이를 현대 AI 기술의 발전과 연관 지어 개인의 자유, 데이터 프라이버시, 알고리즘의 편향성 등 다양한 윤리적 쟁점들을 탐구하고자 합니다. 본론 1: '마이너리티 리포트'의 예측 사법 시스템 분석..." notes="영화 '마이너리티 리포트'의 핵심 줄거리와 메시지를 정확하게 요약해야 합니다.AI 윤리 관련 최신 논의 및 실제 사례들을 리포트에 적절히 통합해야 합니다.자유 의지와 결정론, 데이터 프라이버시, 알고리즘 편향성 등 주요 윤리적 개념들을 명확히 설명해야 합니다.리포트의 결론에서는 영화와 현실 AI 윤리 간의 시사점을 종합하고 미래 방향성을 제시해야 합니다."
💡 ret_result = {'prompt_materials': Object

## 2. Message 포맷의 State 사용하기
State의 저장값으로 Message를 바로 사용하기도 합니다.   
이 경우, Context에 메시지를 계속 추가하거나, 별도의 로직을 만들어 메시지 정보를 전달합니다.
`typing`의 `Annotated`로 공간을 지정한 후, 뒷부분에 결합 로직을 포함합니다.   
이를 리듀서(Reducer)라고 부르는데, 메시지의 경우 아래와 같이 포함하면 됩니다.

## e.g. (3)

In [ ]:
### 예전 코드 --- (start)
import keyring
import os

from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_google_genai import ChatGoogleGenerativeAI

from pydantic import BaseModel, Field

service_name = "gemini-api-key---alpha300uk"
username = "alpha300uk"

# 키링에서 토큰 조회
api_token = keyring.get_password(service_name, username)

# api_token 등록
os.environ['GOOGLE_API_KEY'] = api_token

# Gemini API는 분당 10개 요청으로 제한
# 즉, 초당 약 0.167개 요청 (10/60)
rate_limiter = InMemoryRateLimiter(
    # requests_per_second=0.167,  # 분당 10개 요청
    requests_per_second=1,  # 초당 최대 1개, 분당 최대 60개 요청
    check_every_n_seconds=0.1,  # 100ms마다 체크
    max_bucket_size=10,  # 최대 버스트 크기
)

# rate limiter를 LLM에 적용
llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    rate_limiter=rate_limiter,
    # temperature
    # max_tokens

    thinking_budget = 500  # 추론(Reasoning) 토큰 길이 제한
)
### 예전 코드 --- (end)

### 이번 코드 --- (start)
def talk(State):
    return {'context': AIMessage(content='AI 메시지 2')}

import pprint
## 출력문이 여러 곳에서 동시에 쓰이기에 함수로 공통화
def pprint_talk_graph(graph, input_messages):
    """
    그래프의 'context' 상태에 메시지 리스트를 전달하고
    각 단계별(Node) 실행 결과를 출력하는 헬퍼 함수입니다.
    """
    import pprint
    # 'instruction' 대신 현재 State 구조에 맞는 'context' 키를 사용합니다.
    for data in graph.stream({'context': input_messages}, stream_mode='values'):
        pprint.pprint(data)
        print('----' * 10)


from typing import Annotated
from typing import TypedDict
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    context: Annotated[list, add_messages]

builder = StateGraph(State)
builder.add_node('talk',talk)
builder.add_edge(START, 'talk')
builder.add_edge('talk', END)

graph = builder.compile()

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
messages = [
    SystemMessage(content='시스템 메시지 1'),
    HumanMessage(content='유저 메시지 1'),
    AIMessage(content='AI 메시지 1'),
    HumanMessage(content='유저 메시지 2'),
]

### invoke : 한꺼번에 결과를 낸다.
### stream : 각 노드의 실행이 끝날 때마다 데이터를 순차적으로(Generator) 배출
# response = graph.invoke({'context': messages})
pprint_talk_graph(graph, messages)


### 메시지 삭제 
from langchain_core.messages import RemoveMessage
def delete_message(State):
    # 첫번째,두번째 메시지 삭제
    messages = State['context']
    return {"context": [RemoveMessage(id = messages[i].id) for i in range(1,3)]}


builder = StateGraph(State)

builder.add_node('talk',talk)
builder.add_node('delete_message',delete_message)

builder.add_edge(START, 'talk')
builder.add_edge('talk', 'delete_message')
builder.add_edge('delete_message', END)
graph = builder.compile()

### invoke : 한꺼번에 결과를 낸다.
### stream : 각 노드의 실행이 끝날 때마다 데이터를 순차적으로(Generator) 배출
graph.invoke({'context': messages})
# 유저 메시지 1, AI 메시지 1 삭제
### 이번 코드 --- (end)

{'context': [SystemMessage(content='시스템 메시지 1', additional_kwargs={}, response_metadata={}, id='c61aea47-539a-4431-a141-626aa0494425'),
             HumanMessage(content='유저 메시지 1', additional_kwargs={}, response_metadata={}, id='7b350f26-71f9-4e3d-8282-f601d8d50238'),
             AIMessage(content='AI 메시지 1', additional_kwargs={}, response_metadata={}, id='4c0515b0-ae1f-415a-bf10-6add7bc93afd', tool_calls=[], invalid_tool_calls=[]),
             HumanMessage(content='유저 메시지 2', additional_kwargs={}, response_metadata={}, id='b16b0389-66dc-4633-b2f4-cb9665e8e352')]}
----------------------------------------
{'context': [SystemMessage(content='시스템 메시지 1', additional_kwargs={}, response_metadata={}, id='c61aea47-539a-4431-a141-626aa0494425'),
             HumanMessage(content='유저 메시지 1', additional_kwargs={}, response_metadata={}, id='7b350f26-71f9-4e3d-8282-f601d8d50238'),
             AIMessage(content='AI 메시지 1', additional_kwargs={}, response_metadata={}, id='4c0515b0-ae1f-415a-bf10-

{'context': [SystemMessage(content='시스템 메시지 1', additional_kwargs={}, response_metadata={}, id='c61aea47-539a-4431-a141-626aa0494425'),
  HumanMessage(content='유저 메시지 2', additional_kwargs={}, response_metadata={}, id='b16b0389-66dc-4633-b2f4-cb9665e8e352'),
  AIMessage(content='AI 메시지 2', additional_kwargs={}, response_metadata={}, id='0534ebe9-97d5-4179-8290-aa544f878c9b', tool_calls=[], invalid_tool_calls=[])]}